In [1]:
names = [
    "apple", "red apple", "green apple", "yellow apple",
    "rotten apple", "bruised apple", "moldy apple", "wrinkled apple",
    
]

In [2]:
from ultralytics import YOLOE

model = YOLOE("../models/yoloe-26m-seg.pt")
model.set_classes(names, model.get_text_pe(names))

results = model.predict("../dataset/image1.png", 
                        conf=0.05,
                        imgsz=2000)
results[0].show()

c:\Users\user\Desktop\fruit_detecte\mp\.venv\Lib\site-packages\torch\jit\_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(



WARNING imgsz=[2000] must be multiple of max stride 32, updating to [2016]


FileNotFoundError: ../dataset/image1.png does not exist

In [ ]:
# import os
# import cv2
# import torchvision
# from pathlib import Path
# from ultralytics import YOLOE

# def drop_group_boxes(boxes, min_inside=2, thr=0.8):
#     keep = []
#     for i, a in enumerate(boxes):
#         inside = 0
#         for j, b in enumerate(boxes):
#             if i == j:
#                 continue
#             ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
#             ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
#             inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
#             area_b = (b[2] - b[0]) * (b[3] - b[1])
#             if area_b > 0 and inter / area_b > thr:
#                 inside += 1
#         if inside < min_inside:
#             keep.append(i)
#     return keep


# names = [
#     "apple", "red apple", "green apple", "yellow apple",
#     "rotten apple", "bruised apple", "moldy apple", "wrinkled apple",
# ]

# model = YOLOE("../models/yoloe-26m-seg.pt")
# model.set_classes(names, model.get_text_pe(names))

# INPUT_DIR = "../dataset"     # папка с исходными фото
# OUTPUT_DIR = "../crops"     # куда сохранять кропы
# PAD = 0.05               # отступ вокруг рамки, 5% от её размера
# MIN_SIZE = 20            # кропы меньше 20 px не сохраняем

# os.makedirs(OUTPUT_DIR, exist_ok=True)

# for r in model.predict(INPUT_DIR, conf=0.1, imgsz=2000, stream=True, verbose=False):
#     if len(r.boxes) == 0:
#         continue

#     boxes, confs, clss = r.boxes.xyxy, r.boxes.conf, r.boxes.cls

#     # убираем дубли
#     k = torchvision.ops.nms(boxes, confs, iou_threshold=0.6)
#     boxes, confs, clss = boxes[k], confs[k], clss[k]

#     # убираем рамки-группы
#     k = drop_group_boxes(boxes.tolist())
#     boxes, confs, clss = boxes[k], confs[k], clss[k]

#     img = r.orig_img                     # исходное изображение, BGR
#     h, w = img.shape[:2]
#     stem = Path(r.path).stem

#     for idx, (b, c, cl) in enumerate(zip(boxes, confs, clss)):
#         x1, y1, x2, y2 = b.tolist()
#         bw, bh = x2 - x1, y2 - y1
#         if bw < MIN_SIZE or bh < MIN_SIZE:
#             continue

#         # отступ, чтобы не обрезать края фрукта
#         x1 = max(0, int(x1 - bw * PAD))
#         y1 = max(0, int(y1 - bh * PAD))
#         x2 = min(w, int(x2 + bw * PAD))
#         y2 = min(h, int(y2 + bh * PAD))

#         crop = img[y1:y2, x1:x2]
#         label = r.names[int(cl)].replace(" ", "_")
#         fname = f"{stem}_{idx:03d}_{label}_{float(c):.2f}.png"
#         cv2.imwrite(os.path.join(OUTPUT_DIR, fname), crop)

#     print(f"{stem}: сохранено {len(boxes)} кропов")

In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

In [ ]:
class FruitDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.png')]

        self.class_names = [name for name in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, name))]
        self.class_idx = {name: i for i, name in enumerate(self.class_names)}

        self.sample = []

        for class_name in self.class_names:
            class_dir = os.path.join(root_dir, class_name)
            for img_file in os.listdir(class_dir):
                if img_file.endswith('.png'):
                    self.sample.append((os.path.join(class_dir, img_file), self.class_idx[class_name]))

    def __len__(self):
        return len(self.sample)

    def __getitem__(self, idx):
        img_name, label = self.sample[idx]
        image = Image.open(img_name).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
DATA_DIR = "../dataset_cls/train"      
IMG_SIZE = 128

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),   
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = FruitDataset(DATA_DIR, transform=train_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)

In [ ]:
from collections import Counter

print("Классы:", train_ds.class_names)
print("Количество:", Counter(label for _, label in train_ds.sample))

images, labels = next(iter(train_loader))
print(images.shape, labels[:10])

Классы: ['fresh', 'rotten']
Количество: Counter({1: 98, 0: 30})
torch.Size([32, 3, 128, 128]) tensor([1, 1, 1, 0, 1, 1, 1, 0, 1, 1])


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def predict_proba(self, x):
        return F.softmax(self.forward(x), dim=1)

In [ ]:
model = SimpleCNN(num_classes=len(train_ds.class_names)).to(device)

images, labels = next(iter(train_loader))
out = model(images.to(device))
print(out.shape)          # ожидаем torch.Size([32, 2])

n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n_params:,}")

torch.Size([32, 2])
Параметров: 2,121,122


In [ ]:
from collections import Counter

num_classes = len(train_ds.class_names)

# веса классов против дисбаланса: fresh ≈ 2.13, rotten ≈ 0.65
counts = Counter(label for _, label in train_ds.sample)
total = sum(counts.values())
class_weights = torch.tensor(
    [total / (num_classes * counts[c]) for c in range(num_classes)],
    dtype=torch.float32,
).to(device)
print("Веса классов:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Веса классов: tensor([2.1333, 0.6531])


In [ ]:
EPOCHS = 20
SAVE_PATH = "fruit_cnn.pt"            # NEW: куда сохранять
best_loss = float("inf")              # NEW: лучший loss за всё обучение

for epoch in range(EPOCHS):
    model.train()

    running_loss = 0.0
    correct = torch.zeros(num_classes)
    seen = torch.zeros(num_classes)

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)

        for c in range(num_classes):
            mask = labels == c
            seen[c] += mask.sum().item()
            correct[c] += (preds[mask] == c).sum().item()

    epoch_loss = running_loss / len(train_ds)
    acc = correct.sum() / seen.sum()
    recall = correct / seen

    print(f"Эпоха {epoch+1:2d}/{EPOCHS} | loss {epoch_loss:.4f} | acc {acc:.3f} | "
          f"recall fresh {recall[0]:.3f} | recall rotten {recall[1]:.3f}")

    # NEW: сохраняем, если loss стал лучше
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save({
            "model_state_dict": model.state_dict(),          # веса модели
            "optimizer_state_dict": optimizer.state_dict(),  # состояние Adam (чтобы продолжить обучение)
            "epoch": epoch + 1,
            "loss": epoch_loss,
            "class_names": train_ds.class_names,             # ['fresh', 'rotten']
            "img_size": IMG_SIZE,
        }, SAVE_PATH)
        print(f"  -> сохранено (loss {epoch_loss:.4f})")

Эпоха  1/20 | loss 0.7962 | acc 0.500 | recall fresh 0.367 | recall rotten 0.541
  -> сохранено (loss 0.7962)
Эпоха  2/20 | loss 0.6878 | acc 0.320 | recall fresh 0.967 | recall rotten 0.122
  -> сохранено (loss 0.6878)
Эпоха  3/20 | loss 0.6619 | acc 0.750 | recall fresh 0.733 | recall rotten 0.755
  -> сохранено (loss 0.6619)
Эпоха  4/20 | loss 0.6393 | acc 0.750 | recall fresh 0.333 | recall rotten 0.878
  -> сохранено (loss 0.6393)
Эпоха  5/20 | loss 0.5932 | acc 0.547 | recall fresh 0.900 | recall rotten 0.439
  -> сохранено (loss 0.5932)
Эпоха  6/20 | loss 0.5219 | acc 0.797 | recall fresh 0.600 | recall rotten 0.857
  -> сохранено (loss 0.5219)
Эпоха  7/20 | loss 0.5445 | acc 0.719 | recall fresh 0.800 | recall rotten 0.694
Эпоха  8/20 | loss 0.5295 | acc 0.820 | recall fresh 0.333 | recall rotten 0.969
Эпоха  9/20 | loss 0.4272 | acc 0.734 | recall fresh 0.933 | recall rotten 0.673
  -> сохранено (loss 0.4272)
Эпоха 10/20 | loss 0.4336 | acc 0.688 | recall fresh 0.933 | recall 